## Bandwidth & contrast (functions)
`build_system()` rebuilds the same optical system the cells above set
up inline, and the `propagate_*` functions loop over the filter band and sum the
PSFs *incoherently*.

- `propagate_monochromatic(wl, sys)` — one wavelength through the coronagraph
- `propagate_unocculted(wl, sys)` — on-axis, no-coronagraph reference PSF
- `propagate_bandwidth` / `propagate_unocculted_bandwidth` — incoherent sums over the band
- raw contrast = `PSF_coronagraph / PSF_unocculted_peak`

Usage:

    contrast = run_bandwidth("z", n_wavelengths=11, plot=True)


In [ ]:
import numpy as np
import hcipy as hp
import hcipy as hcipy
from hcipy import *
import matplotlib.pyplot as plt

from utils.magaox import make_magaox_bump_mask
from utils.apodized_mask_generator import make_magaox_piaa_lyot_stop
from utils.PIAACMC import (
    load_fpm_csv, center_fpm_map, make_fpm_apodizer, make_measured_fpm_apodizer,
    DESIGN_RADII, DESIGN_HEIGHTS, F_NUMBER, SPATIAL_RESOLUTION,
)


In [ ]:
import os


FILTERS = {
    "875": {"name": "875 nm", "wavelength": 875e-9, "bandwidth": 0.03},
    "z": {"name": "z band", "wavelength": 908e-9, "bandwidth": 0.13 / 0.908},
}


def build_system(filter_spec):
    """Build every wavelength-independent part of the coronagraph.

    Returns a dict holding the grids, apodizers and propagators that the
    per-wavelength propagation reuses. Nothing here depends on wavelength
    except the refractive-index *functions* (``n(lambda)``), which HCIPy's
    ``SurfaceApodizer`` evaluates at each wavelength for us.
    """
    wavelength = filter_spec["wavelength"]

    # --- telescope pupil (full aperture) ------------------------------------
    
    telescope_diameter = 6.5 # [m], Magellan Clay
    pupil_diameter = 8.604e-3 # [m], bump mask
    pupil_diameter_full = 8.9e-3 # [m], full entrance pupil MagAO-X
    grid_diameter = 1.5 * telescope_diameter # [m], oversize
    F_number = 69
    focal_length = float(F_number * pupil_diameter_full)
    pupil_grid_non_scaled = hp.make_pupil_grid(500, grid_diameter)
    pupil_grid = pupil_grid_non_scaled.scaled(pupil_diameter / telescope_diameter)
    aperture_non_scaled = hp.evaluate_supersampled(make_magaox_bump_mask(), pupil_grid_non_scaled, 8)
    aperture = hp.Field(aperture_non_scaled, pupil_grid)
    piaa_pupil_grid = pupil_grid

    # --- measured PIAA lens profiles and FPM height map ---------------------
    piaa0_csv = np.loadtxt("utils/piaa0_CaF2.csv", delimiter=",", skiprows=1)
    piaa1_csv = np.loadtxt("utils/piaa1_CaF2.csv", delimiter=",", skiprows=1)
    radial_d0, profile0 = piaa0_csv[:, 0], -piaa0_csv[:, 1]
    radial_d1, profile1 = piaa1_csv[:, 0], -piaa1_csv[:, 1]
    profile0 -= np.min(profile0)
    profile1 -= np.min(profile1)

    fpm_height, fpm_pixel_size = load_fpm_csv("utils/FPM.csv")
    fpm_height = center_fpm_map(fpm_height, fpm_pixel_size)
    print("FPM map:", fpm_height.shape, "  pixel size [m]:", fpm_pixel_size)

    # --- PIAA lens surfaces and Fresnel propagators -------------------------
    d_pupil_piaa0 = 37.2e-3   # m, pupil plane -> PIAA0
    d_piaa0_piaa1 = 60.2e-3   # m, PIAA0 -> PIAA1
    # pupil_diameter = 8.9e-3   # m, pupil diameter
    ref_ind_lenses = hp.get_refractive_index("CAF2")   # callable n(lambda)

    # piaa_pupil_grid = hp.make_pupil_grid(500, diameter=1.3 * pupil_diameter)
    # aperture = hp.evaluate_supersampled(make_magaox_bump_mask(aperture_scale="pupil"), piaa_pupil_grid, 8)

    prop_pupil_piaa0 = hp.FresnelPropagator(aperture.grid, d_pupil_piaa0)
    prop_piaa0_piaa1 = hp.FresnelPropagator(aperture.grid, d_piaa0_piaa1)
    prop_piaa1_pupil = hp.FresnelPropagator(aperture.grid, d_piaa0_piaa1 + d_pupil_piaa0)

    prop_LS_piaa1 = hp.FresnelPropagator(aperture.grid, d_pupil_piaa0)
    prop_piaa1_piaa0 = hp.FresnelPropagator(aperture.grid, d_piaa0_piaa1)
    prop_piaa0_LS = hp.FresnelPropagator(aperture.grid, d_piaa0_piaa1 + d_pupil_piaa0)

    sag0 = hp.Field(np.interp(aperture.grid.as_("polar").r, radial_d0, profile0), aperture.grid)
    sag1 = hp.Field(np.interp(aperture.grid.as_("polar").r, radial_d1, profile1), aperture.grid)
    piaa0 = hp.SurfaceApodizer(sag0, ref_ind_lenses)
    piaa1 = hp.SurfaceApodizer(sag1, ref_ind_lenses)

    # --- focal plane, FPM, Lyot stop and detector ---------------------------
    spatial_resolution = SPATIAL_RESOLUTION   # 1 lambda/D at the central wavelength
    fpm_radius = 0.75 * spatial_resolution
    print(f"Focal-plane scale (1 lambda/D) = {spatial_resolution * 1e6:.2f} um,  "
        f"FPM radius = {fpm_radius * 1e6:.2f} um")

    focal_grid = hp.make_focal_grid(q=6, num_airy=16, spatial_resolution=spatial_resolution)
    # cor_grid = hp.make_pupil_grid(500, 150e-6)
    cor_grid = hp.make_pupil_grid(1500, 1000e-6)
    
    n_mask = hp.get_refractive_index("IP_DIP")

    prop_piaa1_focal = hp.FraunhoferPropagator(piaa_pupil_grid, cor_grid, focal_length=focal_length)
    prop_piaa1_focal_big = hp.FraunhoferPropagator(piaa_pupil_grid, focal_grid, focal_length=focal_length)
    prop_fpm_lyot = hp.FraunhoferPropagator(cor_grid, piaa_pupil_grid, focal_length=focal_length)

    # Plain-telescope focal plane (before any coronagraph optics), used only
    # for the broadband telescope-PSF diagnostic.
    prop_telescope = hp.FraunhoferPropagator(aperture.grid, hp.make_focal_grid(q=8, num_airy=16, pupil_diameter=telescope_diameter,
        focal_length=focal_length, reference_wavelength=wavelength), focal_length=focal_length)

    fpm_apodizer = make_measured_fpm_apodizer(fpm_height, fpm_pixel_size, cor_grid, n_mask)
    fpm_apodizer_design = make_fpm_apodizer(DESIGN_RADII, DESIGN_HEIGHTS, cor_grid, n_mask)

    lyot_stop = hp.evaluate_supersampled(
        hp.make_rotated_aperture(make_magaox_piaa_lyot_stop(), np.deg2rad(245 + 90)), piaa_pupil_grid, 8)

    lyot_apodizer = hp.Apodizer(lyot_stop)

    detector_grid = hp.make_focal_grid(
        q=4,
        num_airy=20,
        pupil_diameter=pupil_diameter,
        focal_length=focal_length,
        reference_wavelength=wavelength,
    )
    
    prop_detector = hp.FraunhoferPropagator(aperture.grid, detector_grid, focal_length=focal_length)

    return {
        "filter": filter_spec,
        "pupil_diameter": pupil_diameter,
        "aperture": aperture,
        "focal_length": focal_length,
        "piaa0": piaa0,
        "piaa1": piaa1,
        "lyot_stop": lyot_stop,
        "lyot_apodizer": lyot_apodizer,
        "fpm_apodizer": fpm_apodizer,
        "spatial_resolution": spatial_resolution,
        "prop_pupil_piaa0": prop_pupil_piaa0,
        "prop_piaa0_piaa1": prop_piaa0_piaa1,
        "prop_piaa1_pupil": prop_piaa1_pupil,
        "prop_LS_piaa1": prop_LS_piaa1,
        "prop_piaa1_piaa0": prop_piaa1_piaa0,
        "prop_piaa0_LS": prop_piaa0_LS,
        "prop_piaa1_focal": prop_piaa1_focal,
        "prop_piaa1_focal_big": prop_piaa1_focal_big,
        "prop_fpm_lyot": prop_fpm_lyot,
        "prop_detector": prop_detector,
        "detector_grid": detector_grid,
        "prop_telescope": prop_telescope,
        "radial_d0": radial_d0,
        "profile0": profile0,
        "radial_d1": radial_d1,
        "profile1": profile1,
        "fpm_height": fpm_height,
        "fpm_pixel_size": fpm_pixel_size,
        "fpm_apodizer_design": fpm_apodizer_design,
    }



In [ ]:

def propagate_monochromatic_fields(wavelength, sys, separation=0.0):
    """Propagate one wavelength through the full coronagraph.

    Returns a dict with every intermediate wavefront (and the final PSF
    power) so the diagnostic plots can be rendered without re-running the
    propagation. Keys: ``wf_pupil``, ``wf_apodized``, ``wf_fpm``,
    ``wf_lyot``, ``wf_lyot_stopped``, ``wf_pupil_out``, ``psf_power``.
    """
    aperture = sys["aperture"]
    pupil_diameter = sys["pupil_diameter"]

    # On-axis source unless `separation` (in units of pupil diameter) is set.
    tilt = np.exp(2j * np.pi * separation * aperture.grid.x / pupil_diameter)
    wf_pupil = hp.Wavefront(aperture * tilt, wavelength=wavelength)
    wf_pupil.total_power = 1.0

    # Forward PIAA (apodize).
    wf_piaa0 = sys["piaa0"].forward(sys["prop_pupil_piaa0"].forward(wf_pupil))
    wf_piaa1 = sys["piaa1"].forward(sys["prop_piaa0_piaa1"].forward(wf_piaa0))
    wf_apodized = sys["prop_piaa1_pupil"].backward(wf_piaa1)

    # FPM and Lyot stop.
    wf_fpm = sys["fpm_apodizer"].forward(sys["prop_piaa1_focal"].forward(wf_apodized))
    wf_lyot = sys["prop_fpm_lyot"].forward(wf_fpm)
    wf_lyot_stopped = sys["lyot_apodizer"](wf_lyot)

    # Inverse PIAA (de-apodize back to a round pupil).
    wf_piaa1_out = sys["piaa1"].forward(sys["prop_LS_piaa1"].forward(wf_lyot_stopped))
    wf_piaa0_out = sys["piaa0"].forward(sys["prop_piaa1_piaa0"].forward(wf_piaa1_out))
    wf_pupil_out = sys["prop_piaa0_LS"].backward(wf_piaa0_out)

    psf_power = sys["prop_detector"].forward(wf_pupil_out).power

    return {
        "wf_pupil": wf_pupil,
        "wf_apodized": wf_apodized,
        "wf_fpm": wf_fpm,
        "wf_lyot": wf_lyot,
        "wf_lyot_stopped": wf_lyot_stopped,
        "wf_pupil_out": wf_pupil_out,
        "psf_power": psf_power,
    }


def propagate_monochromatic(wavelength, sys, separation=0.0):
    """Propagate one wavelength through the full coronagraph.

    Returns the monochromatic coronagraphic PSF power (an ``hp.Field`` on the
    detector grid).
    """
    return propagate_monochromatic_fields(wavelength, sys, separation)["psf_power"]


def propagate_unocculted(wavelength, sys, separation=0.0):
    """Reference PSF with the focal-plane mask removed (the 'unocculted' PSF).

    Matches the FPM papers: only the focal-plane mask is taken out of the
    beam; the PIAA lenses and Lyot stop remain. Without the mask the focal
    round-trip is an identity, so the apodised pupil passes straight through
    the (oversized) Lyot stop and is then de-apodised by the inverse PIAA set.
    The coronagraphic PSF is normalised by this PSF's peak to form the raw
    contrast.
    """
    aperture = sys["aperture"]
    pupil_diameter = sys["pupil_diameter"]

    tilt = np.exp(2j * np.pi * separation * aperture.grid.x / pupil_diameter)
    wf_pupil = hp.Wavefront(aperture * tilt, wavelength=wavelength)
    wf_pupil.total_power = 1.0

    # Forward PIAA (apodize).
    wf_piaa0 = sys["piaa0"].forward(sys["prop_pupil_piaa0"].forward(wf_pupil))
    wf_piaa1 = sys["piaa1"].forward(sys["prop_piaa0_piaa1"].forward(wf_piaa0))
    wf_apodized = sys["prop_piaa1_pupil"].backward(wf_piaa1)

    # No focal-plane mask in the beam: the apodised pupil passes the Lyot stop.
    wf_lyot_stopped = sys["lyot_apodizer"](wf_apodized)

    wf_piaa1 = sys["piaa1"].forward(sys["prop_LS_piaa1"].forward(wf_lyot_stopped))
    wf_piaa0 = sys["piaa0"].forward(sys["prop_piaa1_piaa0"].forward(wf_piaa1))
    wf_pupil_out = sys["prop_piaa0_LS"].backward(wf_piaa0)

    return sys["prop_detector"].forward(wf_pupil_out).power


def _sum_over_band(sys, wavelengths, mono_func, spectrum=None):
    """Incoherently sum a per-wavelength PSF ``mono_func(wl, sys)`` over the band."""
    total = None
    for wl in wavelengths:
        weight = 1.0 if spectrum is None else spectrum(wl)
        psf_power = mono_func(wl, sys)
        total = weight * psf_power if total is None else total + weight * psf_power
    return total


In [ ]:
def save_power_figure(power, filename, title, spatial_resolution, unit_label="lambda/D"):
    """Save a normalised, log-scaled PSF image."""
    img = np.log10((power / power.max()))
    fig, ax = plt.subplots(figsize=(6, 5))
    hp.imshow_field(img, cmap="inferno", grid_units=spatial_resolution, vmin=-7, vmax=0)
    ax.set_title(title)
    ax.set_xlabel(f"Detector plane [{unit_label}]")
    ax.set_ylabel(f"Detector plane [{unit_label}]")
    fig.colorbar(ax.images[0], ax=ax, label="Log10(power)")
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)


def _radial_profile(values, grid, spatial_resolution, n_bins=100):
    """Azimuthally average ``values`` on ``grid`` into a radial profile.

    Returns ``(centers, profile)`` where ``centers`` is the annular-bin centre
    in units of ``spatial_resolution`` (lambda/D) and ``profile`` is the mean
    of ``values`` in each bin.
    """
    x = np.asarray(grid.x)
    y = np.asarray(grid.y)
    r = np.hypot(x, y) / spatial_resolution          # radial distance [lambda/D]
    vals = np.asarray(values)

    edges = np.linspace(0, r.max(), n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    profile = np.full(n_bins, np.nan)
    for i in range(n_bins):
        sel = (r >= edges[i]) & (r < edges[i + 1])
        if np.any(sel):
            profile[i] = np.mean(vals[sel])
    return centers, profile


def save_radial_profile(values, grid, filename, title, spatial_resolution,
                        n_bins=100, reference=None):
    """Save the raw-contrast profile (azimuthally averaged) vs lambda/D.

    ``values`` is the already-normalised field to average (e.g. the raw
    contrast ``PSF_coronagraph / PSF_unocculted_peak``). Each annular bin is
    averaged to give the radial contrast curve.

    ``reference`` optionally adds a second curve: the unocculted (non-
    coronagraphic) stellar PSF normalised to 1 at its centre, so the
    coronagraphic suppression can be read off directly against the plain
    telescope PSF.
    """
    centers, profile = _radial_profile(values, grid, spatial_resolution, n_bins)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.semilogy(centers, profile, color="k", lw=1.2, label="Coronagraphic raw contrast")

    if reference is not None:
        _, ref_profile = _radial_profile(reference, grid, spatial_resolution, n_bins)
        ax.semilogy(centers, ref_profile, color="C0", lw=1.2, ls="--",
                    label="Unocculted PSF (normalised)")

    ax.set_title(title)
    ax.set_xlabel(r"Radial distance [$\lambda/D$]")
    ax.set_ylabel("Raw contrast")
    ax.set_ylim(1e-6, 1.0)
    ax.set_xlim(0, 15)
    ax.grid(True, which="both", ls=":", alpha=0.5)
    ax.legend()
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)


def save_diagnostics(sys, out_dir):
    """Save the PIAACMC.ipynb diagnostics as two summary figures.

    ``diagnostics_optics.png`` collects the static optical elements (pupil,
    Lyot stop, PIAA lenses and FPM surfaces); ``diagnostics_light.png``
    collects the light-propagation panels (pupil power, focal planes, PSFs
    and the final-PSF cross-section). Both are rendered once, at the central
    wavelength.
    """
    os.makedirs(out_dir, exist_ok=True)
    wl = sys["filter"]["wavelength"]
    bandwidth = sys["filter"]["bandwidth"]
    name = sys["filter"]["name"]
    spatial_resolution = sys["spatial_resolution"]

    # One representative propagation so the intermediate fields are available.
    fields = propagate_monochromatic_fields(wl, sys)
    wf_apodized = fields["wf_apodized"]
    wf_lyot_stopped = fields["wf_lyot_stopped"]
    psf_power = fields["psf_power"]

    # Reusable, wavelength-independent pieces of the system.
    aperture = sys["aperture"]
    wavefront_focal_before_fpm = sys["prop_piaa1_focal_big"].forward(wf_apodized)
    wf_focal_after_fpm = sys["prop_piaa1_focal_big"].forward(wf_lyot_stopped)

    # Plain-telescope (unapodized) reference PSF, mono + broadband. This is
    # the telescope focal plane (effective focal length 71.5 m), so its plate
    # scale is in um, unlike the coronagraph focal planes (in lambda/D).
    focal_mono = sys["prop_telescope"].forward(hp.Wavefront(aperture, wavelength=wl)).power
    focal_total = 0
    for w in wl * np.linspace(1 - bandwidth / 2.0, 1 + bandwidth / 2.0, 11):
        focal_total = focal_total + sys["prop_telescope"].forward(hp.Wavefront(aperture, wavelength=w)).power

    # ============================ optics figure =============================
    fig, ax = plt.subplots(2, 4, figsize=(22, 11))

    hp.imshow_field(aperture, cmap="gray", grid_units=1e-3, ax=ax[0, 0])
    ax[0, 0].set_title("Telescope pupil (bump mask)")
    ax[0, 0].set_xlabel("x [mm]")
    ax[0, 0].set_ylabel("y [mm]")
    fig.colorbar(ax[0, 0].images[0], ax=ax[0, 0], label="Transmission")

    hp.imshow_field(sys["lyot_stop"], cmap="gray", grid_units=1e-3, ax=ax[0, 1])
    ax[0, 1].set_title("Lyot stop")
    ax[0, 1].set_xlabel("x [mm]")
    ax[0, 1].set_ylabel("y [mm]")
    fig.colorbar(ax[0, 1].images[0], ax=ax[0, 1], label="Transmission")

    hp.imshow_field(sys["piaa0"].surface_sag, cmap="grey", grid_units=1e-3, ax=ax[0, 2])
    ax[0, 2].set_title("PIAA0 surface sag")
    ax[0, 2].set_xlabel("x [mm]")
    ax[0, 2].set_ylabel("y [mm]")
    fig.colorbar(ax[0, 2].images[0], ax=ax[0, 2], label="Sag [m]")

    hp.imshow_field(sys["piaa1"].surface_sag, cmap="grey", grid_units=1e-3, ax=ax[0, 3])
    ax[0, 3].set_title("PIAA1 surface sag")
    ax[0, 3].set_xlabel("x [mm]")
    ax[0, 3].set_ylabel("y [mm]")
    fig.colorbar(ax[0, 3].images[0], ax=ax[0, 3], label="Sag [m]")

    ax[1, 0].plot(sys["radial_d0"] * 1e3, sys["profile0"] * 1e6, label="PIAA0", color="purple")
    ax[1, 0].plot(sys["radial_d1"] * 1e3, sys["profile1"] * 1e6, label="PIAA1", color="hotpink")
    ax[1, 0].set_title("PIAA lens surface profiles")
    ax[1, 0].set_xlabel("Radius [mm]")
    ax[1, 0].set_ylabel("Height [um]")
    ax[1, 0].legend()

    im = ax[1, 1].imshow(sys["fpm_height"], cmap="inferno")
    ax[1, 1].set_title("Measured FPM height map")
    ax[1, 1].set_xlabel("Pixel")
    ax[1, 1].set_ylabel("Pixel")
    fig.colorbar(im, ax=ax[1, 1], label="Height [um]")

    hp.imshow_field(sys["fpm_apodizer_design"].surface_sag * 1e6, cmap="inferno",
                    grid_units=1e-6, ax=ax[1, 2])
    ax[1, 2].set_title("FPM surface (design)")
    ax[1, 2].set_xlabel("x [um]")
    ax[1, 2].set_ylabel("y [um]")
    fig.colorbar(ax[1, 2].images[0], ax=ax[1, 2], label="Sag [um]")

    hp.imshow_field(sys["fpm_apodizer"].surface_sag * 1e6, cmap="inferno",
                    grid_units=1e-6, ax=ax[1, 3])
    ax[1, 3].set_title("FPM surface (measured)")
    ax[1, 3].set_xlabel("x [um]")
    ax[1, 3].set_ylabel("y [um]")
    fig.colorbar(ax[1, 3].images[0], ax=ax[1, 3], label="Sag [um]")

    fig.suptitle(f"{name} — optical components", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(os.path.join(out_dir, "diagnostics_optics.png"), dpi=150)
    plt.close(fig)

    # ============================= light figure =============================
    fig, ax = plt.subplots(2, 4, figsize=(22, 11))

    hp.imshow_field(wf_apodized.power, cmap="gray", grid_units=1e-3, ax=ax[0, 0])
    ax[0, 0].set_title("Apodized pupil (after PIAA)")
    ax[0, 0].set_xlabel("x [mm]")
    ax[0, 0].set_ylabel("y [mm]")
    fig.colorbar(ax[0, 0].images[0], ax=ax[0, 0], label="Power")

    hp.imshow_field(
        np.log10(wavefront_focal_before_fpm.power / wavefront_focal_before_fpm.power.max()),
        cmap="inferno", grid_units=spatial_resolution, vmin=-7, vmax=0, ax=ax[0, 1],
    )
    ax[0, 1].set_title("Focal plane before FPM")
    ax[0, 1].set_xlabel(r"x [$\lambda/D$]")
    ax[0, 1].set_ylabel(r"y [$\lambda/D$]")
    fig.colorbar(ax[0, 1].images[0], ax=ax[0, 1], label="log10(power)")

    hp.imshow_field(
        np.log10(wf_focal_after_fpm.power / wavefront_focal_before_fpm.power.max()),
        cmap="inferno", grid_units=spatial_resolution, vmin=-7, vmax=0, ax=ax[0, 2],
    )
    ax[0, 2].set_title("Focal plane after FPM")
    ax[0, 2].set_xlabel(r"x [$\lambda/D$]")
    ax[0, 2].set_ylabel(r"y [$\lambda/D$]")
    fig.colorbar(ax[0, 2].images[0], ax=ax[0, 2], label="log10(power)")

    hp.imshow_field(wf_lyot_stopped.power / wf_lyot_stopped.power.max(), cmap="gray",
                    grid_units=1e-3, ax=ax[0, 3])
    ax[0, 3].set_title("Lyot plane (after Lyot stop)")
    ax[0, 3].set_xlabel("x [mm]")
    ax[0, 3].set_ylabel("y [mm]")
    fig.colorbar(ax[0, 3].images[0], ax=ax[0, 3], label="Power")

    hp.imshow_field(np.log10(focal_mono / focal_mono.max()), cmap="inferno",
                    grid_units=1e-6, ax=ax[1, 0])
    ax[1, 0].set_title("Telescope PSF (monochromatic)")
    ax[1, 0].set_xlabel("x [um]")
    ax[1, 0].set_ylabel("y [um]")
    fig.colorbar(ax[1, 0].images[0], ax=ax[1, 0], label="log10(power)")

    hp.imshow_field(np.log10(focal_total / focal_total.max()), cmap="inferno",
                    grid_units=1e-6, ax=ax[1, 1])
    ax[1, 1].set_title("Telescope PSF (broadband)")
    ax[1, 1].set_xlabel("x [um]")
    ax[1, 1].set_ylabel("y [um]")
    fig.colorbar(ax[1, 1].images[0], ax=ax[1, 1], label="log10(power)")

    hp.imshow_field(np.log10(psf_power / psf_power.max()), cmap="inferno",
                    grid_units=spatial_resolution, vmin=-7, vmax=0, ax=ax[1, 2])
    ax[1, 2].set_title("Final coronagraphic PSF")
    ax[1, 2].set_xlabel(r"x [$\lambda/D$]")
    ax[1, 2].set_ylabel(r"y [$\lambda/D$]")
    fig.colorbar(ax[1, 2].images[0], ax=ax[1, 2], label="log10(power)")

    two = (psf_power / psf_power.max()).shaped
    avg = np.mean(two, axis=0)
    x_axis = sys["detector_grid"].x.shaped[0, :] / spatial_resolution
    ax[1, 3].semilogy(x_axis, avg, color="k")
    ax[1, 3].set_title("Final PSF cross-section (y-averaged)")
    ax[1, 3].set_xlabel(r"x [$\lambda/D$]")
    ax[1, 3].set_ylabel("Normalized power")
    ax[1, 3].grid(True, which="both", ls=":", alpha=0.5)

    fig.suptitle(f"{name} — light propagation", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(os.path.join(out_dir, "diagnostics_light.png"), dpi=150)
    plt.close(fig)


In [ ]:
def run_bandwidth(filter_name="z", n_wavelengths=11, save=False, plot=True, out_dir="outputs"):
    """Run the coronagraph over a filter band and return the raw-contrast field.

    Returns the raw contrast ``PSF_coronagraph / PSF_unocculted_peak``
    (an ``hp.Field`` on the detector grid).
    """
    filter_spec = FILTERS[filter_name]
    wavelengths = np.linspace(
        filter_spec["wavelength"] * (1 - filter_spec["bandwidth"] / 2),
        filter_spec["wavelength"] * (1 + filter_spec["bandwidth"] / 2),
        n_wavelengths,
    )

    sys = build_system(filter_spec)

    broadband = _sum_over_band(sys, wavelengths, propagate_monochromatic)
    unocculted = _sum_over_band(sys, wavelengths, propagate_unocculted)

    spatial_resolution = sys["spatial_resolution"]

    unocculted_peak = unocculted.max()
    contrast = broadband / unocculted_peak
    unocculted_normalised = unocculted / unocculted_peak

    # Mean raw contrast over 1-5 lambda/D: average the azimuthally-averaged
    # radial profile across that annulus (the metric used in the FPM paper).
    centers, contrast_profile = _radial_profile(
        contrast, sys["detector_grid"], spatial_resolution, n_bins=100
    )
    sel_1_5 = (centers >= 1) & (centers <= 5)
    contrast_1_to_5 = np.nanmean(contrast_profile[sel_1_5])

    print(f"Unocculted PSF peak = {unocculted_peak:.6g}")
    print(f"On-axis raw contrast = {contrast.max():.3e}")
    print(f"1-5 lambda/D raw contrast = {contrast_1_to_5:.3e}")

    if save or plot:
        os.makedirs(out_dir, exist_ok=True)
    if save:
        np.savez(
            os.path.join(out_dir, f"{filter_name}_broadband_psf.npz"),
            power=broadband,
            unocculted_power=unocculted,
            unocculted_peak=unocculted_peak,
            contrast=contrast,
            spatial_resolution=sys["spatial_resolution"],
            wavelengths=wavelengths,
            shape=broadband.grid.shape,
        )
    if plot:
        save_radial_profile(
            contrast, sys["detector_grid"],
            os.path.join(out_dir, f"{filter_name}_raw_contrast.png"),
            f"{filter_spec['name']} — raw contrast",
            sys["spatial_resolution"],
            reference=unocculted_normalised,
        )
        save_power_figure(
            broadband, os.path.join(out_dir, f"{filter_name}_broadband_psf.png"),
            f"{filter_spec['name']} — broadband coronagraphic PSF",
            sys["spatial_resolution"],
        )
        save_diagnostics(sys, out_dir)
        print(f"Saved figures -> {out_dir}/")

    return contrast


In [ ]:
contrast = run_bandwidth(filter_name="z", n_wavelengths=5, save=True, plot=True, out_dir="outputs")
